## ------------------------------------------------------------ ###
## Set up environment
## ------------------------------------------------------------ ###

In [27]:
import os
import sys
import glob
import rasterio
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

sys.path.append('../../ProcessEvents/')
from config import CATCHMENT_LOOKUP_DICT, OUT_DIR # MOLLY_DIR_FF, RAINFALL_CSV_DIR, ENSEMBLE_MEMBERS, , CATCHMENTS
event_details_fp = 'EventDetails_v4' # 'EventDetails_v5'
flood_5km_results_fp = "5km_total" # "5km_total_v5"

## ------------------------------------------------------------ ###
## Get dataframe with all catchment results
## ------------------------------------------------------------ ###

In [39]:
rainfall_events_all = []

for catchment_num in set(CATCHMENT_LOOKUP_DICT.keys()):
    catchment_name = CATCHMENT_LOOKUP_DICT[catchment_num]
    
    ## Get main event data
    fp = os.path.join(OUT_DIR, f"Catchment_{catchment_num}", f"{catchment_name}_old.pkl")
    if not os.path.exists(fp):
        print(fp)
        continue
    events_this_catchment = pd.read_pickle(fp)
    events_this_catchment['catchment_num'] = catchment_num
    events_this_catchment = events_this_catchment.reset_index(drop=True)
    
    ## Get temporal profile data
    tp_fp = f"/scratch/hydro4/users/kv25483/FutureFlood/Data/{event_details_fp}/Catchment_{catchment_num}/TemporalProfiles.csv"
    tp = pd.read_csv(tp_fp)
    tp = tp.reset_index(drop=True)
    
    ## Get landcover and topography data
    dst_fp = f"/scratch/hydro4/users/kv25483/FutureFlood/Data/{event_details_fp}/Catchment_{catchment_num}/DynamicSoilTings.csv"
    dst = pd.read_csv(dst_fp)
    del dst['event_num']
    dst = dst.reset_index(drop=True)

    events_this_catchment = events_this_catchment.loc[:, ~events_this_catchment.columns.duplicated()]
    tp = tp.loc[:, ~tp.columns.duplicated()]
    dst = dst.loc[:, ~dst.columns.duplicated()]    
    
    ## Join
    if not (len(tp) == len(events_this_catchment) == len(dst)):
        raise ValueError(f"Row counts differ: tp={len(tp)}, "f"events={len(events_this_catchment)}, dst={len(dst)}")
        continue
    this_catchment_joined = pd.concat([events_this_catchment, tp, dst],axis=1)
    
    # Add to big dataframe
    this_catchment_joined.reset_index(inplace=True, drop=True)
    rainfall_events_all.append(this_catchment_joined) 
    
rainfall_events_all_df = pd.concat(rainfall_events_all, ignore_index=True)   
len(rainfall_events_all_df.loc[~rainfall_events_all_df["catchment_num"].map(lambda x: isinstance(x, (int, float))), "catchment_num"].unique())

/scratch/hydro4/users/kv25483/FutureFlood/Data/EventDetails_v4/Catchment_26/HullGroup_old.pkl
/scratch/hydro4/users/kv25483/FutureFlood/Data/EventDetails_v4/Catchment_33_b/GreatOuse_b_old.pkl
/scratch/hydro4/users/kv25483/FutureFlood/Data/EventDetails_v4/Catchment_89/AweandEtive_old.pkl


InvalidIndexError: Reindexing only valid with uniquely valued Index objects

### Join the landcover and topography stats

In [36]:
topo_fp = f"/scratch/hydro4/users/kv25483/FutureFlood/Data/{event_details_fp}/topo_and_landcover_stats_for_unique_cells.pkl"
topo = pd.read_pickle(topo_fp)

In [37]:
cols = ["rainpeak_x_global", "rainpeak_y_global"]
rainfall_events_all_df["peak_cell_id"] = rainfall_events_all_df.groupby(cols, sort=False).ngroup()

In [38]:
merged = rainfall_events_all_df.merge(topo.drop_duplicates().assign(matched=True),
    on='peak_cell_id', how='right')

print("Unique topo peak_cell_ids:", topo['peak_cell_id'].nunique())
print("Matched topo peak_cell_ids:", merged.loc[merged['matched'].eq(True), 'peak_cell_id'].nunique())
print("Unmatched topo peak_cell_ids:", merged['matched'].isna().sum())

Unique topo peak_cell_ids: 8929
Matched topo peak_cell_ids: 8929
Unmatched topo peak_cell_ids: 0


## ------------------------------------------------------------ ###
## Check if any catchments have NANs for any variables
## ------------------------------------------------------------ ###

In [16]:
nulls = merged[merged['hc_at_peak'].isnull()]
np.unique(nulls)

array([], dtype=object)

In [17]:
nulls = merged[merged['fu_at_peak_new'].isnull()][['catchment_num', 'ens', 'fu_at_peak_new', 'lu_at_peak', 'event_num']]
nulls

,catchment_num,ens,fu_at_peak_new,lu_at_peak,event_num


## ------------------------------------------------------------ ###
## Check whether any grid cells have multiple events in them
## ------------------------------------------------------------ ###

In [18]:
dupes = merged[merged.duplicated(subset=["rainpeak_x_global", "rainpeak_y_global"],
        keep=False)].sort_values(["rainpeak_x_global", "rainpeak_y_global"])

len(dupes[['start_month', 'lu_at_peak', 'rainpeak_peak_value', 'ens', "rainpeak_x_global", "rainpeak_y_global"]])

106430

## ------------------------------------------------------------ ###
## Save to file
## ------------------------------------------------------------ ###

In [20]:
merged.to_pickle(f"/scratch/hydro4/users/kv25483/FutureFlood/Data/{event_details_fp}/all_catchments.pkl")